> **¿Dónde estamos en el flujo del EDA?**
> En el notebook anterior conocimos los datos y medimos los faltantes (pasos 02 y 04). Ahora avanzamos al paso **05 – Explorar y analizar**, empezando por la **estadística descriptiva** (ver lámina "Medidas estadísticas descriptivas"): tendencia central, dispersión y posición.
> Seguimos con el mismo dataset (*Breast Cancer Wisconsin*): cada fila es una muestra, las columnas son medidas de los núcleos celulares y `target` es el diagnóstico (0 = maligno, 1 = benigno).

In [4]:
#nuevamente importamos pandas
# Cada notebook es independiente: hay que volver a importar librerías y cargar los datos
import pandas as pd

In [5]:
# aplicando el método read_csv(), levantamos el archivo sample_dataset
# Recordatorio: el CSV tiene que estar subido en el panel "Archivos" de ESTE notebook (si no, FileNotFoundError)
df = pd.read_csv("sample_dataset.csv")

In [6]:
#cuántas observaciones totales tiene nuestro conjunto?
# Esperamos (569, 31): 569 filas y 31 columnas, igual que en el notebook anterior
df.shape

(569, 31)

Algo útil sería generar una versión resumida del conjunto de datos que contenga información clave sobre su contenido, lo que implica aplicar funciones de sumarización (análisis descriptivo: suma, promedio, conteo, mínimos, máximos, etc.) para obtener un panorama general del DataFrame.

In [7]:
# describe() calcula las medidas descriptivas de cada columna NUMÉRICA (por defecto ignora las de texto)
# Los NaN no se tienen en cuenta en los cálculos: por eso count puede ser menor a 569
df.describe()
#df.iloc[:, :15].describe() #si quisiera las primeras 15 columnas
# iloc[filas, columnas] selecciona por posición: ":" = todas las filas, ":15" = columnas 0 a 14

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
count,482.000000,492.000000,513.000000,403.000000,384.000000,480.000000,439.000000,382.000000,471.000000,504.000000,...,414.000000,460.000000,512.000000,468.000000,474.000000,507.000000,495.000000,410.000000,470.000000,569.000000
mean,14.059548,19.311829,92.039025,661.522581,0.097156,0.104531,0.094063,0.049115,0.181405,0.062626,...,25.599058,107.322848,893.873828,0.133065,0.256748,0.267936,0.116579,0.291797,0.084363,0.627417
std,3.501791,4.347769,24.028669,356.669534,0.014502,0.053335,0.083301,0.038449,0.027633,0.007102,...,6.186596,33.967766,587.297571,0.023000,0.160373,0.209779,0.066426,0.066327,0.018683,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.490000,54.490000,185.200000,0.071170,0.034320,0.000000,0.000000,0.156600,0.055040,0.000000
25%,11.672500,16.170000,75.270000,428.100000,0.086688,0.064815,0.030410,0.020682,0.161950,0.057530,...,20.842500,84.147500,510.275000,0.117175,0.144500,0.106450,0.065280,0.248725,0.071857,0.000000
50%,13.280000,18.860000,86.340000,556.700000,0.096565,0.093125,0.068240,0.033770,0.179100,0.061300,...,25.225000,97.455000,686.550000,0.131450,0.211850,0.212300,0.101500,0.282050,0.079870,1.000000
75%,15.745000,21.802500,104.700000,796.000000,0.106825,0.130325,0.135100,0.074122,0.196600,0.066003,...,29.485000,126.950000,1141.000000,0.147125,0.343800,0.384100,0.165650,0.319750,0.092172,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.290600,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


1. **count**: número de valores no nulos en la columna.
2. **mean**: promedio de los valores de la columna.
3. **std**: desviación estándar de los valores (qué tan dispersos están respecto del promedio).
4. **min**: valor mínimo encontrado en la columna.
5. **25%**: percentil 25 (Q1); el 25% de los datos son menores o iguales a este valor.
6. **50%**: mediana (Q2); el 50% de los datos son menores o iguales a este valor.
7. **75%**: percentil 75 (Q3); el 75% de los datos son menores o iguales a este valor.
8. **max**: valor máximo encontrado en la columna.

Vemos que trajo 30 columnas (las numéricas) y la categórica (`area error`) no la considera.

**Cómo leer esta tabla rápido:**

- **count vs. 569:** la diferencia son los faltantes que medimos en el notebook anterior.
- **mean vs. 50% (mediana):** si la media es bastante mayor que la mediana, la distribución probablemente tiene **asimetría positiva** (cola a la derecha). Ejemplo: en `mean area` la media queda claramente por encima de la mediana (ver lámina "Distribuciones simétricas y asimétricas").
- **max vs. 75%:** si el máximo está muy lejos del Q3, es una pista de posibles **valores atípicos** (lo vamos a confirmar con boxplots).
- **Escalas distintas:** el std de `mean area` es de cientos y el de `mean smoothness` es de centésimas. No se pueden comparar directamente entre variables.

> **Ojo con `target`:** aparece porque es numérica (`int64`), pero es categórica. Su "media" no es una medida típica: como vale 0 o 1, la media es la **proporción de 1** (casos benignos). Su std, cuartiles, etc. no tienen una interpretación útil.

In [8]:
# Con 30 columnas la tabla queda muy ancha. Transponiéndola con .T, cada variable pasa a ser una fila
df.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,482.0,14.059548,3.501791,6.981000,11.672500,13.280000,15.745000,28.11000
mean texture,492.0,19.311829,4.347769,9.710000,16.170000,18.860000,21.802500,39.28000
mean perimeter,513.0,92.039025,24.028669,43.790000,75.270000,86.340000,104.700000,188.50000
mean area,403.0,661.522581,356.669534,143.500000,428.100000,556.700000,796.000000,2501.00000
mean smoothness,384.0,0.097156,0.014502,0.052630,0.086688,0.096565,0.106825,0.16340
mean compactness,480.0,0.104531,0.053335,0.019380,0.064815,0.093125,0.130325,0.34540
mean concavity,439.0,0.094063,0.083301,0.000000,0.030410,0.068240,0.135100,0.42680
mean concave points,382.0,0.049115,0.038449,0.000000,0.020682,0.033770,0.074122,0.20120
mean symmetry,471.0,0.181405,0.027633,0.106000,0.161950,0.179100,0.196600,0.29060
mean fractal dimension,504.0,0.062626,0.007102,0.049960,0.057530,0.061300,0.066003,0.09744


In [9]:
# Comparemos media y mediana de algunas variables para detectar asimetrías
df[["mean radius", "mean area", "mean smoothness"]].agg(["mean", "median"])

,mean radius,mean area,mean smoothness
mean,14.059548,661.522581,0.097156
median,13.280000,556.700000,0.096565


In [10]:
# Si quisiéramos evaluar otros percentiles, por ejemplo el 95 para evaluar outliers
# pandas los ordena solo y siempre incluye la mediana (50%)
# P5 y P95 muestran dónde está el 90% "central" de los datos: si el min o el max quedan muy lejos de ellos, hay valores extremos
df.describe(percentiles=[0.25,0.5,0.75,0.05,0.95])

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
count,482.000000,492.000000,513.000000,403.000000,384.000000,480.000000,439.000000,382.000000,471.000000,504.000000,...,414.000000,460.000000,512.000000,468.000000,474.000000,507.000000,495.000000,410.000000,470.000000,569.000000
mean,14.059548,19.311829,92.039025,661.522581,0.097156,0.104531,0.094063,0.049115,0.181405,0.062626,...,25.599058,107.322848,893.873828,0.133065,0.256748,0.267936,0.116579,0.291797,0.084363,0.627417
std,3.501791,4.347769,24.028669,356.669534,0.014502,0.053335,0.083301,0.038449,0.027633,0.007102,...,6.186596,33.967766,587.297571,0.023000,0.160373,0.209779,0.066426,0.066327,0.018683,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.490000,54.490000,185.200000,0.071170,0.034320,0.000000,0.000000,0.156600,0.055040,0.000000
5%,9.439400,13.071000,60.922000,279.690000,0.075221,0.041970,0.004926,0.006431,0.140700,0.053783,...,16.767500,67.878000,337.815000,0.097336,0.071879,0.014069,0.025448,0.210315,0.062888,0.000000
25%,11.672500,16.170000,75.270000,428.100000,0.086688,0.064815,0.030410,0.020682,0.161950,0.057530,...,20.842500,84.147500,510.275000,0.117175,0.144500,0.106450,0.065280,0.248725,0.071857,0.000000
50%,13.280000,18.860000,86.340000,556.700000,0.096565,0.093125,0.068240,0.033770,0.179100,0.061300,...,25.225000,97.455000,686.550000,0.131450,0.211850,0.212300,0.101500,0.282050,0.079870,1.000000
75%,15.745000,21.802500,104.700000,796.000000,0.106825,0.130325,0.135100,0.074122,0.196600,0.066003,...,29.485000,126.950000,1141.000000,0.147125,0.343800,0.384100,0.165650,0.319750,0.092172,1.000000
95%,20.569000,27.700000,135.780000,1307.100000,0.121755,0.210745,0.255150,0.120275,0.234550,0.076020,...,36.094500,171.145000,2020.350000,0.171910,0.574745,0.678210,0.240170,0.412880,0.120455,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.290600,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


In [11]:
#si queremos una sumarización de variables categóricas, podemos pasarla como parámetro
# include=["object", "category"] le pide a describe() solo las columnas de texto/categóricas
# Tip: include="all" muestra todas juntas (aparecen NaN donde una medida no aplica, por ejemplo la media de un texto)
df.describe(include=["object","category"])

,area error
count,494
unique,3
top,A
freq,489


Observamos que la descripción recibida es diferente a la que se logra con variables numéricas.

En el output se muestra un resumen estadístico de la variable categórica llamada "area error" el cual incluye:

* count: 494, lo que indica el número de valores no nulos.
* unique: 3, que representa el número de valores únicos.
* top: "A", el valor más frecuente.
* freq: 489, la frecuencia con la que aparece el valor más común ("A").

#### Esto sugiere que la mayoría de los registros en esta columna tienen el valor "A".

> **Para pensar:** si 489 de 494 valores son "A", B y C aparecen solo 5 veces en total. Una variable casi constante aporta muy poca información para diferenciar casos. Además, como vimos en el notebook anterior, esta columna fue modificada para la clase: en el dataset original es numérica.

In [12]:
#si queremos ver la cuenta de cada valor que asume la variable categórica "area error" :
# Por defecto value_counts() NO cuenta los NaN: con dropna=False los incluye
# Con normalize=True obtenemos proporciones en lugar de cantidades
df['area error'].value_counts()

,count
area error,
A,489
B,4
C,1


**¿Y la variable objetivo?** Antes de cerrar, conviene mirar cómo se reparte `target`: ¿hay la misma cantidad de casos malignos que benignos? Esto se llama **balance de clases** y va a importar mucho cuando comparemos grupos o armemos modelos.

In [13]:
# 0 = maligno, 1 = benigno. normalize=True muestra la proporción de cada clase
df["target"].value_counts(), df["target"].value_counts(normalize=True)

(target
 1    357
 0    212
 Name: count, dtype: int64,
 target
 1    0.627417
 0    0.372583
 Name: proportion, dtype: float64)

En este notebook indagamos un paso más allá de los rasgos generales del Dataset y empezamos a ver un poco cómo se distribuyen los datos, valores medios, medianas, percentiles, máximos, mínimos y características de variables categóricas